In [1]:
import pandas as pd
import geopandas as gpd
import json

#show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Merge BLS data with AI scores

In [2]:
#define the study rating you want to use. we'll programmatically see if there is an
#xwalk file for that study rating. If not you'll get an error and you gotta figure that shit out
study_rating = 'human_rating_beta'

#bring in the AI scores merged with our custom crosswalk from 01_clean_crosswalk.ipynb
occ_scores = pd.read_csv(f'../data/processed/study_scores_xwalk_merged_{study_rating}.csv')
occ_scores['nem_merge'] = occ_scores['NEM Code'].str.replace('-','')

# BLS employment estimates
oews = pd.read_csv('../data/processed/bls_oews_current_employment.csv', dtype={'occupation_code': str, 'area_code': str, 'datatype_code': str})

#let's also define an outfile for the next steps
outfile = f'../data/processed/bls_occ_employment_w_study_scores_{study_rating}.csv'

## Some brief checks
To make sure everything is as we think it is

In [3]:
#see which columns have NAs
print(oews.isna().sum())

series_id                       0
year                            0
period                          0
areatype_code                   0
areatype_name                   0
state_code                      0
area_code                       0
area_name                       0
occupation_code                 0
soc_code                      583
occupation_name                 0
employment                   5010
footnote_codes             231789
series_title                    0
is_all_occupations              0
has_released_employment         0
dtype: int64


In [4]:
#All of the empty employment values have footnote == 8 which means that the data weren't released for the occupation
display(oews.loc[oews['employment'].isna()].sample(2))
display(oews.loc[(oews['employment'].isna())].groupby('footnote_codes',dropna=False).size().reset_index(name='count').sort_values('count', ascending=False))

,series_id,year,period,areatype_code,areatype_name,state_code,area_code,area_name,occupation_code,soc_code,occupation_name,employment,footnote_codes,series_title,is_all_occupations,has_released_employment
227836,OEUS720000000000019501201,2025,A01,S,State,72,7200000,Puerto Rico,195012,19-5012,Occupational Health and Safety Technicians,NaN,8.0,Employment for Occupational Health and Safety Technicians in All Industries in Puerto Rico,False,False
178584,OEUM004590000000043601201,2025,A01,M,Metropolitan or nonmetropolitan area,26,0045900,"Traverse City, MI",436012,43-6012,Legal Secretaries and Administrative Assistants,NaN,8.0,"Employment for Legal Secretaries and Administrative Assistants in All Industries in Traverse City, MI",False,False


,footnote_codes,count
0,8.0,5010


In [5]:
#all of the empty soc codes are "All Occupations"
display(oews.loc[oews['soc_code'].isna()].sample(2))
display(oews.loc[(oews['soc_code'].isna())].groupby('occupation_code',dropna=False).size().reset_index(name='count').sort_values('count', ascending=False))

,series_id,year,period,areatype_code,areatype_name,state_code,area_code,area_name,occupation_code,soc_code,occupation_name,employment,footnote_codes,series_title,is_all_occupations,has_released_employment
71976,OEUM002614000000000000001,2025,A01,M,Metropolitan or nonmetropolitan area,12,0026140,"Homosassa Springs, FL",000000,NaN,All Occupations,35580.0,NaN,"Employment for All Occupations in All Industries in Homosassa Springs, FL",True,True
46941,OEUM002058000000000000001,2025,A01,M,Metropolitan or nonmetropolitan area,48,0020580,"Eagle Pass, TX",000000,NaN,All Occupations,18820.0,NaN,"Employment for All Occupations in All Industries in Eagle Pass, TX",True,True


,occupation_code,count
0,000000,583


## Clean up and merge

In [6]:
#let's get rid of data that doesn't have employment numbers cause we don't need those
oews = oews.loc[oews['employment'].notna()]

#make sure we don't have duplicates in the occ_scores dataframe, the nem_merge is the important
#one to check cause that's what we're merging on. just curious about soc_code duplicates too
print(occ_scores.duplicated(subset=['nem_merge']).sum())
print(occ_scores.duplicated(subset=['soc_code']).sum())

#merge with our xwalk
oews_with_scores = oews.merge(occ_scores, left_on='occupation_code', right_on='nem_merge', how='left', suffixes=('_oews', '_study'))

#clean it up
keep_cols =  ['series_id', 'year', 'period', 'areatype_code',
              'state_code', 'area_code', 'area_name', 'occupation_code',
              'occupation_name', 'employment', 'study_rating',
              'footnote_codes', 'is_all_occupations', 'has_released_employment',
              'NEM Code','nem_merge','soc_code_study']

oews_with_scores = oews_with_scores[keep_cols]

0
14


## More checks
Specifically, we wanna look for any BLS data that doesn't have a matched score.

In [7]:
#which occupations don't have a study rating? let's see if we can figure out why
key_market = (oews_with_scores['area_name'].str.contains('San Francisco'))
study_missing = (oews_with_scores['study_rating'].isna())
general_occ = (~oews_with_scores['occupation_code'].str.endswith('0000'))
not_all_other = (~oews_with_scores['occupation_name'].str.contains('All Other'))

missing_market_records = oews_with_scores.loc[study_missing&general_occ&key_market]
print(len(missing_market_records))
display(missing_market_records.sort_values('employment',ascending=False).head(5))

41


,series_id,year,period,areatype_code,state_code,area_code,area_name,occupation_code,occupation_name,employment,study_rating,footnote_codes,is_all_occupations,has_released_employment,NEM Code,nem_merge,soc_code_study
146459,OEUM004186000000025909901,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",259099,"Educational Instruction and Library Workers, All Other",5740.0,NaN,NaN,False,True,NaN,NaN,NaN
146438,OEUM004186000000025119901,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",251199,"Postsecondary Teachers, All Other",4190.0,NaN,NaN,False,True,NaN,NaN,NaN
146450,OEUM004186000000025309901,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",253099,"Teachers and Instructors, All Other",3180.0,NaN,NaN,False,True,NaN,NaN,NaN
146666,OEUM004186000000041909901,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",419099,"Sales and Related Workers, All Other",3010.0,NaN,NaN,False,True,NaN,NaN,NaN
146716,OEUM004186000000043919901,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",439199,"Office and Administrative Support Workers, All Other",2910.0,NaN,NaN,False,True,NaN,NaN,NaN


In [8]:
#dup soc_code is just our fake soc_code meant to indicate that the study score is an aggregation
occ_scores.groupby('soc_code').size().reset_index(name='count').sort_values('count', ascending=False).head(10)

,soc_code,count
770,"avg from multi-match, all other",15
578,49-2091.00,1
508,45-2041.00,1
509,45-2091.00,1
510,45-2092.00,1
511,45-2093.00,1
512,45-3031.00,1
513,45-4011.00,1
514,45-4021.00,1
515,45-4022.00,1


In [9]:
#make sure our choice metros don't have dup occupations
print(len(oews), 'rows in the original table')
print(len(oews_with_scores), 'rows in the merged table')

markets = ['San Francisco-Oakland-Fremont, CA','Albany-Schenectady-Troy, NY','Houston-Pasadena-The Woodlands, TX']

choice_oews_with_scores = oews_with_scores.loc[oews_with_scores['area_name'] == markets[0]]
choice_oews_with_scores.groupby(['occupation_code']).size().reset_index(name='count').sort_values('count', ascending=False).head(10)

231789 rows in the original table
231789 rows in the merged table


,occupation_code,count
0,000000,1
469,434181,1
461,434081,1
462,434111,1
463,434121,1
464,434131,1
465,434141,1
466,434151,1
467,434161,1
468,434171,1


In [10]:
display(oews_with_scores.head())
print(oews_with_scores.columns)

,series_id,year,period,areatype_code,state_code,area_code,area_name,occupation_code,occupation_name,employment,study_rating,footnote_codes,is_all_occupations,has_released_employment,NEM Code,nem_merge,soc_code_study
0,OEUM001018000000000000001,2025,A01,M,48,0010180,"Abilene, TX",000000,All Occupations,75070.0,NaN,NaN,True,True,NaN,NaN,NaN
1,OEUM001018000000011000001,2025,A01,M,48,0010180,"Abilene, TX",110000,Management Occupations,5470.0,NaN,NaN,False,True,NaN,NaN,NaN
2,OEUM001018000000011101101,2025,A01,M,48,0010180,"Abilene, TX",111011,Chief Executives,40.0,0.350000,NaN,False,True,11-1011,111011,11-1011.00
3,OEUM001018000000011102101,2025,A01,M,48,0010180,"Abilene, TX",111021,General and Operations Managers,2120.0,0.384615,NaN,False,True,11-1021,111021,11-1021.00
4,OEUM001018000000011202101,2025,A01,M,48,0010180,"Abilene, TX",112021,Marketing Managers,140.0,0.578125,NaN,False,True,11-2021,112021,11-2021.00


Index(['series_id', 'year', 'period', 'areatype_code', 'state_code',
       'area_code', 'area_name', 'occupation_code', 'occupation_name',
       'employment', 'study_rating', 'footnote_codes', 'is_all_occupations',
       'has_released_employment', 'NEM Code', 'nem_merge', 'soc_code_study'],
      dtype='str')


## Categorization and export!

In [11]:
#categorize
def get_ai_exposure_category(score):
    if score <= .25:
        return 'Low'
    elif score <= .5:
        return 'Medium low'
    elif score <= .75:
        return 'Medium high'
    elif score > .75:
        return 'High'
    else:
        return 'NA'

oews_with_scores['ai_exposure_category'] = oews_with_scores['study_rating'].apply(get_ai_exposure_category)

#and since we want the employment categories to be based on the area they're in, let's do a qcut for each area
oews_with_scores['employment_category'] = oews_with_scores.groupby('area_name')['employment'].transform(lambda x: pd.qcut(x, q=[0, .25, .5, .75, 1], labels=['Low', 'Medium low', 'Medium high', 'High']))

#and lastly I want to add in the top-level occupation for each detailed occupation
with open('../data/raw/series_id_codes.json', 'r') as f:
    codes = json.load(f)
top_level_occ_names = codes['occupation_codes']['major_occupational_groups']
oews_with_scores['top_occ_code'] = oews_with_scores['occupation_code'].str.slice(0, 2) + '0000'
oews_with_scores['top_occ_name'] = oews_with_scores['top_occ_code'].map(top_level_occ_names)

#export for use in the next notebook
oews_with_scores.to_csv(outfile, index=False)